<a href="https://colab.research.google.com/github/isurika23/EN4554_Deep-Learning-for-Vision/blob/main/EN4554_Assignment02_CNN/EN4554_Assignment02_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup


In [3]:
import random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# fix random seeds
def set_seed(seed=42):
    random.seed(seed);
    np.random.seed(seed)
    torch.manual_seed(seed);
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


Dataset: CIFAR-10 (torchvision.datasets.CIFAR10) <br/>
Training set: 45k | Validation set: 5k

In [ ]:
raw = torchvision.datasets.CIFAR10('./data', train=True, download=True)

 25%|██▍       | 42.1M/170M [08:43<25:51, 82.7kB/s]

In [ ]:
g = torch.Generator().manual_seed(42)
perm = torch.randperm(50000, generator=g)
train_idx = perm[:45000].tolist()
val_idx   = perm[45000:].tolist()

# Per-channel statistics from the TRAINING split only
x = torch.tensor(raw.data[train_idx]).float() / 255.0   # shape (45000, 32, 32, 3)
mean = x.mean(dim=(0, 1, 2))
std  = x.std(dim=(0, 1, 2))
print("mean:", mean, "std:", std)

In [ ]:
normalize = T.Normalize(mean.tolist(), std.tolist())
base_tf = T.Compose([T.ToTensor(), normalize])

def get_loaders(train_tf=base_tf, batch_size=128, n_train=None):
    # Two dataset objects, so augmentation applies to train only, never to val
    train_set = torchvision.datasets.CIFAR10('./data', train=True, transform=train_tf)
    val_set   = torchvision.datasets.CIFAR10('./data', train=True, transform=base_tf)
    idx = train_idx if n_train is None else train_idx[:n_train]
    train_loader = DataLoader(Subset(train_set, idx), batch_size=batch_size, shuffle=True,
                              num_workers=2, generator=torch.Generator().manual_seed(42))
    val_loader = DataLoader(Subset(val_set, val_idx), batch_size=512,
                            shuffle=False, num_workers=2)
    return train_loader, val_loader